[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/kepler/kepler.ipynb)

# Through periapsis with a position spinor

A highly eccentric orbit crosses periapsis so quickly that a fixed physical-time step can miss the turn. Carry the position as a scaled spinor and let physical time advance by the orbit's radius, the distance from the attracting centre at the origin: the inverse-square motion becomes a smooth oscillator. We integrate both descriptions and measure their errors against the exact bound orbit.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.animation import save_figure, show_animation
from examples.mechanics.kepler import render
from examples.mechanics.kepler.core import State, BoundOrbit

In [ ]:
ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Spinor = ga.gatype.even()

In [ ]:
gravity = 1.0
semimajor_axis = 1.0
eccentricities = np.array([0.6, 0.9, 0.97])
featured_case = 2
lift_samples = 64

# Start at apoapsis. The spinor times its reverse, spinor.scalar_norm_squared(), is the distance
# from the attracting centre at the origin, the orbit's radius; its sandwich, spinor >> mv.x, is the position.
apoapsis_radius = semimajor_axis * (1 + eccentricities)                  # [cases]
speed = np.sqrt(gravity / semimajor_axis * (1 - eccentricities) / (1 + eccentricities))  # [cases]
spinor = mv.rotor() * np.sqrt(apoapsis_radius)                           # [cases] Spinor
velocity = mv.y * speed                                                  # [cases] Vector
energy = velocity.scalar_norm_squared() / 2 - gravity / spinor.scalar_norm_squared()  # [cases] Scalar
# Lift the physical velocity without introducing rotation about the radial direction.
rate = velocity * spinor * mv.x / 2                                      # [cases] Spinor
orbit = BoundOrbit(spinor, rate, energy)                                 # [cases] BoundOrbit


def spinor_at(orbit: BoundOrbit, parameter: Scalar) -> Spinor:
    """Exact oscillator evolution of the position spinor."""
    phase = orbit.frequency * parameter                                  # [...] Scalar
    return orbit.spinor * phase.cos() + orbit.rate * (phase.sin() / orbit.frequency)


# Equal regularized steps resolve periapsis. One physical orbit is half a spinor cycle.
parameter = np.linspace(0, np.pi, lift_samples + 1)[:, None] / orbit.frequency  # [lift_samples + 1, cases] Scalar
spinors = spinor_at(orbit, parameter)                                    # [lift_samples + 1, cases] Spinor
positions = spinors >> mv.x                                              # [lift_samples + 1, cases] Vector
save_figure(render.draw_lift(positions[:, featured_case], spinors[:, featured_case]), "kepler_spinor_lift");

The spinor turns and scales the fixed reference direction `mv.x`; its extra freedom is rotation about that direction, which leaves position unchanged. Negative orbital energy makes its regularized acceleration proportional to the spinor itself, with a negative coefficient. The physical clock advances by the radius, `spinor.scalar_norm_squared()`, so an equal regularized step covers much less physical time near periapsis. The points above are equally spaced in that clock. The left panel is the orbit in the xy plane, the right the spinor’s scalar and xy parts.

In the Kustaanheimo–Stiefel matrix notation, with $u \in \mathbb R^4$ the spinor's four components and $s$ the regularized clock, the position reads as $(\mathbf x, 0) = L(u)\,u$ with the $4\times4$ KS matrix $L(u)$, the clock as $dt/ds = u^\mathsf{T} u$, the motion as $d^2u/ds^2 = \tfrac{E}{2}\,u$, and the lift without rotation about the reference direction as the bilinear condition $\ell(u, du/ds) = 0$.

In [ ]:
steps = 128
reference_samples = 1024
period = 2 * np.pi * np.sqrt(semimajor_axis ** 3 / gravity)
placement = Spinor >> mv.x                                               # Vector <- Spinor, Spinor


def physical_state(spinor: Spinor, rate: Spinor, time: Scalar) -> State:
    """Read a position spinor and its regularized derivative in physical time."""
    radius = spinor.scalar_norm_squared()                                # [...] Scalar
    # The sandwich differentiated in both slots; their trivector parts, the unused gauge rate, cancel.
    velocity = (placement(rate, spinor) + placement(spinor, rate)) / radius  # [...] Vector
    return State(time, spinor >> mv.x, velocity)


def exact_orbit(orbit: BoundOrbit, parameter: Scalar) -> State:
    """Exact oscillator evolution and its integrated physical clock, for bound motion."""
    phase = orbit.frequency * parameter                                  # [...] Scalar
    sine = phase.sin()                                                   # [...] Scalar
    current_rate = orbit.rate * phase.cos() - orbit.spinor * (orbit.frequency * sine)  # [...] Spinor
    # Physical time advances by the radius: the clock slows smoothly near the centre.
    initial_radius = orbit.spinor.scalar_norm_squared()                  # [...] Scalar
    rate_radius = orbit.rate.scalar_norm_squared() / orbit.frequency.squared()  # [...] Scalar
    overlap = orbit.spinor.scalar_product(orbit.rate.reverse()) / orbit.frequency  # [...] Scalar
    time = ((initial_radius + rate_radius) * parameter / 2
            + (initial_radius - rate_radius) * (2 * phase).sin() / (4 * orbit.frequency)
            + overlap * sine.squared() / orbit.frequency)                # [...] Scalar
    return physical_state(spinor_at(orbit, parameter), current_rate, time)


def physical_verlet(initial: State, gravity: float, step: float, steps: int) -> Iterator[State]:
    """Kick-drift-kick integration with equal steps in physical time."""
    position, velocity, time = initial.position, initial.velocity, initial.time  # [...] Vector, Vector, Scalar
    acceleration = -gravity * position / position.norm() ** 3            # [...] Vector
    yield initial
    for _ in range(steps):
        half_velocity = velocity + acceleration * (step / 2)             # [...] Vector
        position = position + half_velocity * step                       # [...] Vector
        acceleration = -gravity * position / position.norm() ** 3        # [...] Vector
        velocity = half_velocity + acceleration * (step / 2)             # [...] Vector
        time = time + step                                               # [...] Scalar
        yield State(time, position, velocity)


def regularized_verlet(orbit: BoundOrbit, time: Scalar, step: Scalar, steps: int) -> Iterator[State]:
    """Kick-drift-kick integration of the spinor oscillator from a start time, with its physical clock."""
    spinor, rate = orbit.spinor, orbit.rate                              # [...] Spinor each
    yield physical_state(spinor, rate, time)
    for _ in range(steps):
        half_rate = rate + orbit.energy * spinor * (step / 4)            # [...] Spinor
        following = spinor + half_rate * step                            # [...] Spinor
        # Integrate the radius, spinor.scalar_norm_squared(), by Simpson's rule over the drift.
        time = time + step / 6 * (spinor.scalar_norm_squared()
                                  + (spinor + following).scalar_norm_squared()
                                  + following.scalar_norm_squared())     # [...] Scalar
        rate = half_rate + orbit.energy * following * (step / 4)         # [...] Spinor
        spinor = following
        yield physical_state(spinor, rate, time)


start_time = mv.scalar() * np.zeros_like(apoapsis_radius)                # [cases] Scalar
parameter = np.linspace(0, np.pi, reference_samples + 1)[:, None] / orbit.frequency  # [reference_samples + 1, cases] Scalar
reference = exact_orbit(orbit, parameter)                                # [reference_samples + 1, cases] State
initial = physical_state(spinor, rate, start_time)                       # [cases] State
physical = State.collect(physical_verlet(initial, gravity, period / steps, steps))  # [steps + 1, cases] State
regularized = State.collect(regularized_verlet(orbit, start_time, np.pi / orbit.frequency / steps, steps))  # [steps + 1, cases] State
save_figure(render.draw_comparison(reference, physical, regularized), "kepler_passage");

The columns are the eccentricities 0.6, 0.9 and 0.97. Both numerical paths use kick–drift–kick Verlet and the same step count. The spinor oscillator has a constant coefficient instead of an inverse-cube acceleration factor; its clock concentrates physical-time resolution near the close passage. The lower row enlarges periapsis. This is a fixed-step comparison: an adaptive physical-time solver could also resolve the encounter. Equal step counts do not imply equal wall-clock cost, and regularization does not make the numerical solution exact.

In [ ]:
counts = np.array([64, 128, 256, 512])
clock_iterations = 44


def at_time(orbit: BoundOrbit, time: Scalar, iterations: int) -> State:
    """Invert the monotone physical clock by batched bisection."""
    mean_radius = (orbit.spinor.scalar_norm_squared()
                   + orbit.rate.scalar_norm_squared() / orbit.frequency.squared()) / 2  # [...] Scalar
    middle = time / mean_radius                                          # [...] Scalar
    half_period = np.pi / orbit.frequency                                # [...] Scalar
    lower, upper = middle - half_period, middle + half_period            # [...] Scalar each
    for _ in range(iterations):
        middle = (lower + upper) / 2                                     # [...] Scalar
        before = exact_orbit(orbit, middle).time < time                  # [...] bool
        lower = before * middle + (1 - before) * lower                   # [...] Scalar
        upper = before * upper + (1 - before) * middle                   # [...] Scalar
    return exact_orbit(orbit, (lower + upper) / 2)


def position_error(orbit: BoundOrbit, trajectory: State) -> Scalar:
    # Invert the exact clock by bisection, comparing at each method's physical sample times.
    exact = at_time(orbit, trajectory.time, clock_iterations)            # [samples, ...] State
    squared = (trajectory.position - exact.position).scalar_norm_squared()  # [samples, ...] Scalar
    intervals = trajectory.time[1:] - trajectory.time[:-1]               # [samples - 1, ...] Scalar
    # Weight by elapsed physical time so clustered periapsis samples do not dominate the norm.
    integral = ((squared[1:] + squared[:-1]) * intervals / 2).sum(axis=0)  # [...] Scalar
    return (integral / (trajectory.time[-1] - trajectory.time[0])).square_root() / semimajor_axis  # [...] Scalar


physical_errors = stack([position_error(orbit, State.collect(physical_verlet(
    initial, gravity, period / count, count))) for count in counts])     # [counts, cases] Scalar
regularized_errors = stack([position_error(orbit, State.collect(regularized_verlet(
    orbit, start_time, np.pi / orbit.frequency / count, count))) for count in counts])  # [counts, cases] Scalar
save_figure(render.draw_errors(counts, physical_errors, regularized_errors), "kepler_convergence");

The error is a physical-time-weighted RMS position difference, measured against the analytic orbit at the numerical trajectory’s own times. Both schemes converge quadratically when resolved; the eccentric cases expose how much resolution a fixed physical clock needs. Both schemes keep the angular momentum `position ^ velocity`; the regularized one still has an energy error, which swings out at periapsis and returns over the orbit.

In [ ]:
animation_frames = 120
duration_ms = 60
revolutions = 2

parameter = np.linspace(0, revolutions * np.pi, animation_frames + 1)[:, None] / orbit.frequency  # [animation_frames + 1, cases] Scalar
spinors = spinor_at(orbit, parameter)                                    # [animation_frames + 1, cases] Spinor
positions = spinors >> mv.x                                              # [animation_frames + 1, cases] Vector
show_animation(render.animate_lift(positions[:, featured_case], spinors[:, featured_case]),
               "kepler_spinor_lift", duration_ms)

The animation advances uniformly in the regularized clock, not physical time. The orbit closes twice while the spinor closes once: opposite spinors carry the reference direction to the same position. The construction applies in three spatial dimensions; the displayed orbit lies in the xy plane. This example treats bound, unperturbed Newtonian motion away from exact collision; physical velocity remains singular at collision even though the spinor equation is regular. See [Hestenes and Lounesto, *Geometry of Spinor Regularization*](https://davidhestenes.net/misc/Hestenes_1983_-_Geometry_of_Spinor_Regularization.pdf).